In [ ]:
# Install required packages
!pip install -q transformers torch accelerate sentencepiece protobuf pandas matplotlib plotly

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

### Part 1: Major Open-Weight Model Families (May 2026)

The open-weight LLM ecosystem has expanded dramatically. As of May 2026, [Artificial Analysis](https://artificialanalysis.ai/leaderboards/models) tracks 366+ currently ranked models in its live leaderboard snapshot, and the best open-weight models remain close enough to proprietary leaders to be viable for serious production work.

Key developments:
- **Kimi K2.6 and MiMo-V2.5-Pro** still define the top open-weight quality tier (both score 54)
- **DeepSeek V4 Pro / V4 Flash** remain the easiest shorthand for quality/value open deployment
- **Qwen split into multiple practical branches**: general reasoning (`Qwen3.5` / `Qwen3.6`), coding (`Qwen3 Coder Next`), multimodal live assistants (`Qwen3 Omni`), and vision-language (`Qwen3 VL`)
- **OpenAI gpt-oss models** changed the low-cost hosted inference conversation, even if they are not the most open releases by transparency
- **Openness is now a first-class dimension**: strong open-weight quality and high documentation transparency are no longer the same thing
- **Gemma 4, Nemotron, Granite, and OLMo** matter for specific deployment constraints even when they are not the absolute top by intelligence

The table below focuses on practical model choices rather than trying to list every model family release. Use it to decide which ecosystems are worth deeper study and which branches solve distinct problems.

In [ ]:
# Comprehensive open-weight model database (May 2026)
# Directional data from artificialanalysis.ai live leaderboard snapshot
import pandas as pd

models_data = {
    'Model': [
        # Frontier open-weight
        'Kimi K2.6', 'MiMo-V2.5-Pro', 'DeepSeek V4 Pro (Max)',
        'Muse Spark (Meta)', 'Qwen3.6 Plus', 'GLM-5.1',
        'MiniMax-M2.7', 'DeepSeek V4 Flash (Max)',
        # Strong open-weight
        'Qwen3.5 397B-A17B', 'DeepSeek V3.2', 'Gemma 4 31B',
        'NVIDIA Nemotron 3 Super', 'gpt-oss-120B (high)',
        # Smaller / efficient
        'Mistral Small 4', 'gpt-oss-20B (high)',
        'Gemma 4 E4B', 'Qwen3.5 4B', 'Phi-4',
    ],
    'Intelligence_Index': [
        54, 54, 52,
        52, 50, 51,
        50, 47,
        45, 42, 39,
        36, 33,
        28, 24,
        19, 27, 10,
    ],
    'Parameters': [
        'Unknown', '~50B+', '685B MoE',
        'Unknown', 'Unknown', 'Unknown',
        'Unknown', '685B MoE',
        '397B (17B active)', '685B MoE', '31B',
        '120B (12B active)', '120B',
        'Small', '20B',
        '~4B', '4B', '14B',
    ],
    'Context_Length': [
        256000, 1000000, 1000000,
        262000, 1000000, 200000,
        205000, 1000000,
        262000, 128000, 256000,
        1000000, 131000,
        256000, 131000,
        128000, 262000, 16000,
    ],
    'Price_per_1M': [
        0.70, 0.71, 0.18,
        0, 0.43, 0.90,
        0.22, 0.06,
        0.90, 0.32, 0.17,
        0.28, 0.20,
        0.20, 0.07,
        0, 0.04, 0.16,
    ],
    'Speed_tok_s': [
        68, 61, 47,
        0, 53, 65,
        55, 117,
        52, 82, 35,
        157, 327,
        169, 243,
        0, 178, 18,
    ],
    'License': [
        'Open', 'Open', 'MIT',
        'Meta', 'Apache 2.0', 'Open',
        'Open', 'MIT',
        'Apache 2.0', 'MIT', 'Gemma ToU',
        'Open', 'OpenAI OSS',
        'Apache 2.0', 'OpenAI OSS',
        'Gemma ToU', 'Apache 2.0', 'MIT',
    ],
    'Commercial_Use': [
        'Check terms', 'Check terms', 'Yes',
        'Check terms', 'Yes', 'Check terms',
        'Check terms', 'Yes',
        'Yes', 'Yes', 'Yes (T&C)',
        'Yes', 'Check terms',
        'Yes', 'Check terms',
        'Yes (T&C)', 'Yes', 'Yes',
    ],
}

df_models = pd.DataFrame(models_data)
df_models = df_models.sort_values('Intelligence_Index', ascending=False)

print("Open-Weight Model Landscape (May 2026)")
print("Data source: artificialanalysis.ai live leaderboard snapshot")
print("=" * 80)
print(df_models.to_string(index=False))
print()
print("Current live leaderboard snapshot: 366+ ranked models")
print("Top open-weight score: 54 (Kimi K2.6, MiMo-V2.5-Pro)")
print("Top overall score: 60 (GPT-5.5 xhigh - proprietary)")

In [ ]:
# Visualize model performance vs size using the current Intelligence Index dataset
# Extract the first numeric parameter count when available
df_models['Param_Numeric'] = df_models['Parameters'].str.extract('(\d+)').astype(float)

plot_df = df_models.dropna(subset=['Param_Numeric']).copy()

fig = px.scatter(
    plot_df,
    x='Param_Numeric',
    y='Intelligence_Index',
    size='Context_Length',
    color='License',
    hover_data=['Model', 'Commercial_Use'],
    log_x=True,
    title='Open-Weight Models: Intelligence vs Size (May 2026)',
    labels={'Param_Numeric': 'Parameters (Billions)', 'Intelligence_Index': 'Intelligence Index'}
)
fig.update_layout(height=600)
fig.show()

print("\n📊 Key Insights:")
print("- Larger models often perform better, but efficient families still compress capability well")
print("- Kimi, MiMo, and DeepSeek define the current open-weight quality tier")
print("- Smaller Gemma, Qwen, and Phi variants remain useful for constrained local deployment")
print("- Context length varies dramatically across open-weight families")

## Part 2: Model Family Deep Dives

### Qwen Family (Alibaba)

The Qwen ecosystem is now broad enough that learners should think of it as several product lines, not one model series.

- **Qwen3.5 / Qwen3.6**: the main general-purpose open family for reasoning and deployment across many sizes
- **Qwen3 Next**: newer reasoning-oriented hosted/open-family direction worth tracking for high-end Alibaba releases
- **Qwen3 Coder Next**: coding-specialized branch for agentic coding and software tasks
- **Qwen3 Omni**: multimodal branch aimed at speech, audio, and live assistant style use cases
- **Qwen3 VL**: vision-language branch for image understanding and multimodal reasoning

**Why it matters**: if you teach Qwen only as a single text model family, learners miss how one vendor now covers general LLMs, coding models, voice-first assistants, and VL systems under a unified ecosystem.

### Kimi / MiMo / DeepSeek - The Quality Open-Weight Tier

These are the families most learners should recognize as the current open-weight quality frontier.

- **Kimi K2.6**: best current open-weight quality leader on the live leaderboard
- **MiMo-V2.5-Pro**: ties Kimi at the top while offering long-context practicality
- **DeepSeek V4 Pro / V4 Flash**: strongest quality/value pair to remember for open deployment

**Practical takeaway**: when learners ask "what are the best open models right now?", these are the first names they should know before diving into narrower families.

### OLMo / Nemotron / Granite - The Openness and Systems Tier

These families matter even when they are not the raw intelligence leaders.

- **OLMo 3 family**: among the strongest examples of genuinely open and transparent releases
- **NVIDIA Nemotron 3 Super / Nano V2**: useful when you care about inference speed, openness, and NVIDIA ecosystem alignment
- **IBM Granite 4.1 / 4.0 H family**: important for enterprise-oriented, smaller, or highly efficient deployments

**Practical takeaway**: some families are worth learning because they are easier to reproduce, inspect, license, or serve economically, not because they top every benchmark.

### Gemma / Phi / Smaller Local Models

Smaller families still matter because learners often start on constrained hardware.

- **Gemma 4 E2B / E4B / 31B**: strong Google-backed options across small and larger local setups
- **Phi-4 / Phi-4 Mini / Phi-4 Multimodal**: still useful for small, focused deployments
- **Granite 4.1 8B / 3B**: practical enterprise-leaning smaller models worth knowing

### Family Selection Heuristic

- Learn **Qwen** if you want one ecosystem spanning general, coding, omni, and VL branches.
- Learn **Kimi / MiMo / DeepSeek** if you care most about current open-weight quality.
- Learn **OLMo / Nemotron / Granite** if openness, documentation, and systems concerns matter.
- Learn **Gemma / Phi** if you care about smaller local deployment and fine-tuning accessibility.

In [ ]:
# Compare top models by use case
use_cases = {
    'Use Case': [
        'General Chat', 'Code Generation', 'Reasoning',
        'Edge Deployment', 'Privacy-Sensitive', 'Long Context',
        'Multilingual', 'Fine-tuning Base'
    ],
    'Best Model': [
        'Kimi K2.6', 'Qwen3 Coder Next', 'MiMo-V2.5-Pro',
        'Gemma 4 E4B', 'OLMo 3', 'MiMo-V2.5-Pro',
        'Qwen3.6 Plus', 'Granite 4.1 8B'
    ],
    'Alternative': [
        'DeepSeek V4 Pro', 'DeepSeek V4 Pro', 'DeepSeek V4 Pro',
        'Qwen3.5 4B', 'Nemotron 3 Super', 'Kimi K2.6',
        'GLM-5.1', 'Gemma 4 E4B'
    ],
    'Why': [
        'Strongest open-weight quality for general assistant tasks',
        'Current coding-oriented Qwen branch to track',
        'Top-tier reasoning with long-context practicality',
        'Small enough to run locally while staying useful',
        'Transparency and self-hosting matter more than API convenience',
        'One of the strongest current open long-context choices',
        'Broad multilingual coverage in a widely deployed family',
        'Smaller enterprise-friendly base for adaptation and serving'
    ]
}

df_use_cases = pd.DataFrame(use_cases)
print("\n🎯 Model Selection by Use Case:\n")
df_use_cases

### Part 3: Licensing Considerations

Licensing is one of the most overlooked aspects of open-source model selection, yet it determines whether you can legally deploy a model in your product. The spectrum ranges from fully permissive licenses (Apache 2.0, MIT) that allow unrestricted commercial use, to restrictive licenses that impose revenue limits or prohibit using model outputs to train competing models. Always review the specific license terms before committing to a model for production use -- switching models later due to licensing issues is far more expensive than choosing correctly upfront.

In [ ]:
license_comparison = {
    'License': [
        'Apache 2.0', 'MIT', 'Open-weight custom', 'Gemma Terms',
        'OpenAI OSS', 'Meta license'
    ],
    'Commercial_Use': [
        'Yes', 'Yes', 'Check model terms', 'Yes (T&C apply)',
        'Check model terms', 'Check model terms'
    ],
    'Modifications': [
        'Yes', 'Yes', 'Usually yes', 'Yes',
        'Yes', 'Yes'
    ],
    'Redistribution': [
        'Yes', 'Yes', 'Varies by release', 'With attribution',
        'Varies by release', 'Varies by release'
    ],
    'Revenue_Limit': [
        'None', 'None', 'Varies', 'None',
        'None stated in this summary', 'None stated in this summary'
    ],
    'Examples': [
        'Qwen 3.5/3.6, Mistral Small 4', 'DeepSeek V4, Phi-4',
        'Kimi K2.6, MiMo-V2.5-Pro, GLM-5.1', 'Gemma 4 family',
        'gpt-oss-120B / gpt-oss-20B', 'Muse Spark and other Meta-released weights'
    ]
}

df_licenses = pd.DataFrame(license_comparison)
print("License Comparison:\n")
df_licenses

### ⚠️ Important License Notes

**Apache 2.0 & MIT:**
- ✅ Most permissive
- ✅ No usage restrictions
- ✅ Recommended for startups

**Llama 2:**
- ⚠️ Free if < 700M monthly active users
- ⚠️ Need special license from Meta if larger
- ⚠️ Cannot use outputs to improve other LLMs

**Llama 3:**
- ✅ Removed MAU restriction
- ✅ More permissive than Llama 2
- ⚠️ Still has some restrictions

**Always read the full license before deployment!**

### Part 4: Performance Benchmarks (May 2026)

The [Artificial Analysis Intelligence Index](https://artificialanalysis.ai/methodology/intelligence-benchmarking) has become one of the most useful practical references for comparing modern models. It combines hard evaluations across agents, coding, general reasoning, and scientific reasoning into one score.

**Key benchmarks in the Intelligence Index:**
- **GDPval-AA** - real-world work tasks across 44 occupations
- **τ²-Bench Telecom** - dual-control agent-user simulation
- **Terminal-Bench Hard** - terminal-based task execution
- **SciCode** - scientific Python code generation
- **AA-Omniscience** - knowledge reliability and hallucination
- **Humanity's Last Exam** - frontier academic reasoning
- **GPQA Diamond** - PhD-level science questions
- **CritPt** - research-level physics reasoning

> **Note**: legacy benchmarks like MMLU, HellaSwag, TruthfulQA, and HumanEval are still useful context, but they no longer separate frontier and top open-weight models the way newer agentic and hallucination-aware benchmarks do.

In [ ]:
# Intelligence Index scores for open-weight models (May 2026)
# Source: artificialanalysis.ai live leaderboard snapshot

import plotly.express as px

benchmark_data = {
    'Model': [
        'Kimi K2.6', 'MiMo-V2.5-Pro', 'DeepSeek V4 Pro',
        'Muse Spark', 'Qwen3.6 Plus', 'GLM-5.1',
        'MiniMax-M2.7', 'DeepSeek V4 Flash',
        'Qwen3.5 397B', 'DeepSeek V3.2', 'Gemma 4 31B',
        'Nemotron 3 Super', 'gpt-oss-120B',
    ],
    'Intelligence_Index': [54, 54, 52, 52, 50, 51, 50, 47, 45, 42, 39, 36, 33],
    'Price_per_1M': [0.70, 0.71, 0.18, 0, 0.43, 0.90, 0.22, 0.06, 0.90, 0.32, 0.17, 0.28, 0.20],
    'Speed_tok_s': [68, 61, 47, 0, 53, 65, 55, 117, 52, 82, 35, 157, 327],
    'License': [
        'Open', 'Open', 'MIT', 'Meta', 'Apache 2.0', 'Open',
        'Open', 'MIT', 'Apache 2.0', 'MIT', 'Gemma ToU', 'Open', 'OpenAI OSS',
    ],
}

df_bench = pd.DataFrame(benchmark_data)

fig = px.bar(
    df_bench.sort_values('Intelligence_Index', ascending=True),
    x='Intelligence_Index',
    y='Model',
    color='License',
    orientation='h',
    title='Open-Weight Model Intelligence Index (May 2026, artificialanalysis.ai)',
    labels={'Intelligence_Index': 'Intelligence Index Score', 'Model': ''},
)
fig.update_layout(height=500)
fig.show()

df_priced = df_bench[df_bench['Price_per_1M'] > 0]
fig2 = px.scatter(
    df_priced,
    x='Price_per_1M',
    y='Intelligence_Index',
    text='Model',
    size='Speed_tok_s',
    color='License',
    title='Intelligence vs Price: Open-Weight Models (May 2026)',
    labels={'Price_per_1M': 'Price (USD/1M tokens, blended)', 'Intelligence_Index': 'Intelligence Index'},
)
fig2.update_traces(textposition='top center')
fig2.update_layout(height=500)
fig2.show()

print("\nKey findings:")
print("  - DeepSeek V4 Flash is still one of the strongest value models")
print("  - MiniMax-M2.7 remains a notable budget-quality option")
print("  - gpt-oss-120B combines low price with unusually high output speed")

### Part 5: Model Selection Framework

Choosing the right model requires balancing hardware, license constraints, context length, cost, and target capability. The `select_model()` helper below filters the May 2026 open-weight dataset and returns the top candidates sorted by **Intelligence Index**, which is a better current default than older single-benchmark scores. In practice, you should still validate on your own workload because benchmark rankings do not fully predict domain-specific quality.

In [ ]:
def select_model(use_case, constraints):
    """Return a filtered shortlist from the current open-weight model table."""

    max_size = constraints.get('max_size', '70B')
    license_req = constraints.get('license', 'any').lower()
    min_context = constraints.get('min_context', 0)
    need_commercial = constraints.get('commercial', True)

    filtered = df_models.copy()

    max_size_num = float(max_size.replace('B', ''))
    filtered = filtered[filtered['Param_Numeric'].fillna(float('inf')) <= max_size_num]
    filtered = filtered[filtered['Context_Length'] >= min_context]

    if license_req != 'any':
        filtered = filtered[filtered['License'].str.lower().str.contains(license_req, na=False)]

    if need_commercial:
        filtered = filtered[
            filtered['Commercial_Use'].str.contains('Yes|Check', case=False, na=False)
        ]

    filtered = filtered.sort_values(
        ['Intelligence_Index', 'Price_per_1M', 'Speed_tok_s'],
        ascending=[False, True, False]
    )

    use_case = use_case.lower()
    if use_case == 'code':
        print("\nCode recommendation:")
        print("Track Qwen3 Coder Next first, then DeepSeek V4 Pro for strong coding capability.")
    elif use_case == 'reasoning':
        print("\nReasoning recommendation:")
        print("Start with MiMo-V2.5-Pro or Kimi K2.6, then compare against DeepSeek V4 Pro.")
    elif use_case == 'edge':
        print("\nEdge deployment recommendation:")
        print("Start with Gemma 4 E4B, Qwen3.5 4B, or other small open-weight variants.")
    elif use_case == 'chat':
        print("\nChat application recommendation:")
        print("Start with Kimi K2.6 or DeepSeek V4 Pro, then route cheaper traffic to smaller models.")

    print("\nModels matching your constraints (top 5):\n")
    return filtered.head(5)[[
        'Model', 'Parameters', 'Intelligence_Index', 'Price_per_1M', 'License', 'Context_Length'
    ]]

# Example: select an Apache-licensed chat-capable model under 13B parameters
result = select_model(
    use_case='chat',
    constraints={
        'max_size': '13B',
        'license': 'apache',
        'min_context': 8000,
        'commercial': True
    }
)
result

### Part 6: Deployment Considerations

The gap between "model works in a notebook" and "model serves production traffic" is bridged by understanding **hardware requirements**, **quantization trade-offs**, and **cost economics**. A 70B parameter model in FP16 needs 140GB of VRAM -- far beyond any single consumer GPU -- but 4-bit quantization reduces this to 35GB, making it feasible on a single A100. The cost comparison between local deployment, cloud hosting, and proprietary APIs reveals that open-source models become dramatically cheaper at high volumes (10M+ tokens per month), while proprietary APIs are simpler and more cost-effective at low volumes.

In [ ]:
# Hardware requirements
hardware_reqs = {
    'Model Size': ['2-3B', '7B', '13B', '34B', '70B'],
    'Min_VRAM_FP16': ['6 GB', '16 GB', '32 GB', '70 GB', '140 GB'],
    'Min_VRAM_8bit': ['4 GB', '8 GB', '16 GB', '35 GB', '70 GB'],
    'Min_VRAM_4bit': ['2 GB', '4 GB', '8 GB', '18 GB', '35 GB'],
    'Typical_Device': [
        'Laptop GPU', 'Consumer GPU (RTX 3090)', 
        'High-end GPU (RTX 4090)', 'A100 40GB', 
        'Multi-GPU or A100 80GB'
    ],
    'Tokens_per_sec': ['50-100', '20-40', '10-20', '5-10', '2-5']
}

df_hw = pd.DataFrame(hardware_reqs)
print("Hardware Requirements by Model Size:\n")
df_hw

### 💡 Optimization Techniques

**Quantization:**
- FP16: Half precision, ~50% size reduction
- 8-bit: ~75% size reduction, minimal quality loss
- 4-bit (GPTQ/GGUF): ~87.5% size reduction, some quality loss

**Other Optimizations:**
- Flash Attention: Faster inference
- LoRA: Efficient fine-tuning
- vLLM: Optimized serving
- llama.cpp: CPU-optimized inference

**Trade-offs:**
- Lower precision = faster, less memory, slight quality drop
- Smaller models = faster, less capable
- Always test your specific use case!

In [ ]:
# Cost comparison (directional May 2026 examples for deployment choices)
cost_data = {
    'Deployment': [
        'Gemma 4 E4B (local)', 'Qwen3.5 4B (cloud)',
        'DeepSeek V4 Flash (hosted)', 'Kimi K2.6 (hosted)',
        'gpt-oss-20B (hosted)', 'GPT-5.5 (API)'
    ],
    'Setup_Cost': [0, 0, 0, 0, 0, 0],
    'Monthly_Cost_Low': [0, 20, 30, 120, 25, 150],
    'Monthly_Cost_High': [0, 150, 350, 1200, 220, 4000],
    'Cost_per_1M_tokens': [0, 0.04, 0.06, 0.70, 0.07, 4.35],
    'Control': ['Full', 'High', 'Medium', 'Low', 'Medium', 'None'],
    'Privacy': ['Max', 'High', 'Medium', 'Low', 'Medium', 'Low']
}

df_cost = pd.DataFrame(cost_data)
print("\nDeployment Cost Comparison (Monthly USD, directional):\n")
print(df_cost.to_string(index=False))

print("\nDecision Factors:")
print("\nChoose local open-weight serving if:")
print("  - Privacy, customization, or offline access are top priorities")
print("  - You can keep GPU or edge infrastructure busy enough to justify operations")

print("\nChoose hosted open-weight models if:")
print("  - You want low-cost serving with less infrastructure work")
print("  - You need routing layers, classifiers, or budget assistants")

print("\nChoose proprietary frontier APIs if:")
print("  - You need best-in-class capability immediately")
print("  - You can accept higher token costs for difficult requests")

## Key Takeaways (May 2026)

1. **Open-weight quality is now a real production option** - Kimi K2.6 and MiMo-V2.5-Pro sit only one tier below the absolute frontier leaders.
2. **DeepSeek V4 Flash and V4 Pro remain the easiest quality/value pair to remember** for open deployment.
3. **Qwen is now several families, not one** - general models, coding models, omni models, and VL models all matter.
4. **Open weights and real openness are different** - OLMo and similar releases matter because transparency now has its own benchmark story.
5. **API provider choice is critical** - the same model can be much faster or cheaper depending on where you host it.
6. **The best learning strategy is family-level understanding** - know which ecosystems dominate quality, which dominate openness, and which dominate cost efficiency.